# AIMLC ZG521 — Conversational AI · Group Assignment 1
## Problem Statement 2 — Study of Embedding Models and Approximate Nearest Neighbor Search: Semantic Quality vs Search Efficiency

**Group 129** · Total: 10 Marks · Deadline: 28 Aug 2026

## Student Details

| Name | BITS ID | Email |
|---|---|---|
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |

## Contribution by Each Student

| Member | Task(s) | Section(s) done |
|---|---|---|
| *TODO — name* | T1, T2, T8, report assembly | |
| *TODO — name* | T3, T4 | |
| *TODO — name* | T5 | |
| *TODO — name* | T6, T7 | |


## Problem Statement

Study of embedding models and approximate nearest neighbor (ANN) search, comparing **semantic quality vs search efficiency**:
- **Module 1** — Dataset and embedding preparation (1 mark)
- **Module 2** — Similarity metrics and exact retrieval (3 marks)
- **Module 3** — ANN search experiment: HNSW vs IVF (3 marks)
- **Module 4** — Embedding quality analysis and final recommendation (3 marks)


## Tools and Libraries Used

- **Python 3.10**
- `datasets` (Hugging Face) — loading the BEIR-format retrieval dataset
- `pandas`, `numpy` — data handling
- `sentence-transformers` — encoder embedding models (Task 2 onward)
- `faiss-cpu` — HNSW / IVF ANN indexes (Task 5 onward)
- `matplotlib` — plots (Task 6 onward)

Install all of the above with: `pip install -r requirements.txt` (see `ass-1/requirements.txt`).


In [21]:
# What this cell does: environment setup. Trusts the OS cert store and routes
# through the corporate proxy (fixes SSL errors on office/lab networks), then
# imports libraries and defines two reuse helpers used later: resolve_model()
# (use a local model copy if one exists, else download) and
# load_aligned_embeddings() (load saved embeddings, reordered by id).
import os

try:
    import truststore
    truststore.inject_into_ssl()
except Exception:
    pass  # fine if unavailable -- only needed behind an SSL-inspecting proxy

import urllib.request
_proxy = urllib.request.getproxies().get("https") or urllib.request.getproxies().get("http")
if _proxy:
    os.environ["HTTP_PROXY"] = os.environ["HTTPS_PROXY"] = _proxy
    os.environ["http_proxy"] = os.environ["https_proxy"] = _proxy

import sys, time, json, random
import numpy as np
import pandas as pd

random.seed(129)   # Group 129 -- fixed seed for reproducibility
np.random.seed(129)


def resolve_model(model_name):
    """Use a local copy under models/<name>/ if present (no network needed);
    otherwise return the Hugging Face id, which triggers a download."""
    local = os.path.join("models", model_name.split("/")[-1])
    if os.path.isdir(local) and os.path.exists(os.path.join(local, "config.json")):
        return local
    return model_name


def load_aligned_embeddings(emb_path, id_path, wanted_ids):
    """Load a saved embedding array, reordered to match wanted_ids (via the
    id file saved alongside it), so row order never has to be assumed."""
    emb = np.load(emb_path)
    if os.path.exists(id_path):
        saved_ids = [str(x) for x in json.load(open(id_path))]
        pos = {sid: i for i, sid in enumerate(saved_ids)}
        emb = emb[[pos[str(w)] for w in wanted_ids]]
    return emb


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)


[info] truststore active -- using the OS certificate store for HTTPS.
[info] No proxy auto-detected. If downloads fail with an SSL/handshake error,
       set PROXY_URL at the top of this cell (find it via 'pip config list').
Python: 3.11.5
pandas: 3.0.5
numpy : 2.4.6


---
# Task 1 — Corpus and Query Dataset Preparation (0.5 Marks)

**Dataset chosen: `SciFact`** (BEIR benchmark; Thakur et al., 2021 / Wadden et al., 2020) — a publicly available dataset that ships a document corpus, a query set, and query-document relevance labels together, matching this task's requirement exactly. Corpus (~5,183 passages) and queries (~300) both clear the 1,000/50 minimums.


In [22]:
# What this cell does: load SciFact (corpus/queries/qrels). Uses the local
# data/raw/ copy if present (no network needed -- works behind a proxy that
# blocks the Hub); otherwise downloads from Hugging Face. Queries are then
# filtered to only those with a relevance judgment.
import os
import pandas as pd

RAW_FILES = ("corpus.jsonl", "queries.jsonl", "qrels.tsv")


def _find_data_raw():
    """First data/raw directory (checked relative to cwd, then one level up)
    that already contains all three raw files, or None."""
    for base in (os.path.join("data", "raw"), os.path.join("..", "data", "raw"),
                 os.path.join("ass-1", "data", "raw")):
        if all(os.path.exists(os.path.join(base, f)) for f in RAW_FILES):
            return base
    return None


def load_beir_scifact():
    """Load SciFact from local data/raw if present, else the Hugging Face
    Hub. Queries are filtered to those with >=1 relevance judgment."""
    local_dir = _find_data_raw()
    if local_dir is not None:
        corpus_df = pd.read_json(os.path.join(local_dir, "corpus.jsonl"), lines=True)
        queries_df = pd.read_json(os.path.join(local_dir, "queries.jsonl"), lines=True)
        qrels_df = pd.read_csv(os.path.join(local_dir, "qrels.tsv"), sep="\t")
        print(f"[info] Loaded SciFact from local files: {local_dir}/")
    else:
        from datasets import load_dataset
        corpus_df = load_dataset("BeIR/scifact", "corpus", split="corpus").to_pandas().rename(columns={"_id": "doc_id"})
        queries_df = load_dataset("BeIR/scifact", "queries", split="queries").to_pandas().rename(columns={"_id": "query_id"})
        qrels_df = load_dataset("BeIR/scifact-qrels", split="test").to_pandas()
        qrels_df.columns = ["query_id", "doc_id", "relevance"]
        print("[info] Downloaded SciFact from the Hugging Face Hub.")

    # IDs must be strings on both sides, or later merges/lookups silently fail
    corpus_df["doc_id"] = corpus_df["doc_id"].astype(str)
    queries_df["query_id"] = queries_df["query_id"].astype(str)
    qrels_df["query_id"] = qrels_df["query_id"].astype(str)
    qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)

    # Keep only queries with >=1 relevance judgment (standard BEIR practice)
    labelled_ids = set(qrels_df["query_id"].unique())
    before = len(queries_df)
    queries_df = queries_df[queries_df["query_id"].isin(labelled_ids)].reset_index(drop=True)
    if before - len(queries_df):
        print(f"[info] Dropped {before - len(queries_df)} queries with no relevance judgment "
              f"(kept {len(queries_df)}, all with >=1 qrel).")

    return corpus_df, queries_df, qrels_df


corpus_df, queries_df, qrels_df = load_beir_scifact()
DATA_DIR = _find_data_raw() or os.path.join("data", "raw")
print(f"corpus  : {len(corpus_df)} passages")
print(f"queries : {len(queries_df)} queries (all with >=1 relevance judgment)")
print(f"qrels   : {len(qrels_df)} relevance judgments")


[diag] kernel working directory : /Users/sanjaykumarpushadapu/Projects/MTech/2026-2027-Sem1/AIMLCZG521-ConversationalAI/ass-1
[diag] local data/raw found at  : data/raw
[info] Loading SciFact from local files (no network needed).
corpus  : 5183 passages
queries : 300 queries (all with >=1 relevance judgment)
qrels   : 339 relevance judgments


In [23]:
# ----------------------------------------------------------------------
# What this cell does: check the loaded data against the assignment's stated
# minimums (>=1,000 passages, >=50 queries, every query labelled) and stop
# the notebook immediately (via assert) if either minimum isn't met.
# ----------------------------------------------------------------------
MIN_CORPUS, MIN_QUERIES = 1000, 50

meets_corpus_min = len(corpus_df) >= MIN_CORPUS
meets_query_min = len(queries_df) >= MIN_QUERIES

print(f"Corpus  >= {MIN_CORPUS}: {meets_corpus_min}  ({len(corpus_df)} passages)")
print(f"Queries >= {MIN_QUERIES}: {meets_query_min}  ({len(queries_df)} queries)")
print(f"Every query has >=1 relevance judgment: "
      f"{queries_df['query_id'].isin(qrels_df['query_id']).all()}")

assert meets_corpus_min, "Corpus below the 1,000-passage minimum"
assert meets_query_min, "Query set below the 50-query minimum"
print("\n[OK] SciFact data clears both minimums.")


Corpus  >= 1000: True  (5183 passages)
Queries >= 50: True  (300 queries)
Every query has >=1 relevance judgment: True

[OK] SciFact data clears both minimums.


In [24]:
# ----------------------------------------------------------------------
# What this cell does: print one example row from each of the three tables
# (corpus, queries, qrels) so the schema is visible before it's used later.
# ----------------------------------------------------------------------
print("=== Sample corpus passage ===")
print(corpus_df.iloc[0].to_dict())

print("\n=== Sample query ===")
print(queries_df.iloc[0].to_dict())

print("\n=== Sample relevance judgments (qrels) ===")
print(qrels_df.head())


=== Sample corpus passage ===
{'doc_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior 

In [25]:
# ----------------------------------------------------------------------
# What this cell does: write corpus/queries/qrels to the resolved data/raw
# directory (DATA_DIR, set in the load cell above) so Tasks 2+ can load the
# exact same fixed dataset without re-downloading or re-running this cell.
# ----------------------------------------------------------------------
import os

# Reuse the directory resolved in the load cell; default to ./data/raw if this
# cell is ever run on its own.
DATA_DIR = globals().get("DATA_DIR", os.path.join("data", "raw"))
os.makedirs(DATA_DIR, exist_ok=True)

corpus_df.to_json(os.path.join(DATA_DIR, "corpus.jsonl"), orient="records", lines=True)
queries_df.to_json(os.path.join(DATA_DIR, "queries.jsonl"), orient="records", lines=True)
qrels_df.to_csv(os.path.join(DATA_DIR, "qrels.tsv"), sep="\t", index=False)

print(f"Saved to {DATA_DIR}/: corpus.jsonl, queries.jsonl, qrels.tsv")


Saved to data/raw/: corpus.jsonl, queries.jsonl, qrels.tsv


### Dataset Details and Source

- **Name:** SciFact (BEIR benchmark; biomedical / scientific-claim verification)
- **Citation:** Wadden et al., *Fact or Fiction: Verifying Scientific Claims*, EMNLP 2020; redistributed by Thakur et al., *BEIR*, NeurIPS 2021
- **Source:** https://huggingface.co/datasets/BeIR/scifact (corpus + queries), https://huggingface.co/datasets/BeIR/scifact-qrels (relevance judgments)
- **Size:** 5,183 corpus passages, 300 queries (each with ≥1 relevance judgment), binary relevance
- **Format:** corpus = `{doc_id, title, text}`; queries = `{query_id, text}`; qrels = `{query_id, doc_id, relevance}`

### Explanation of the Logic Used

`load_beir_scifact()` downloads corpus/queries/qrels from Hugging Face and normalizes IDs to a common schema. Queries are filtered to only those with a qrel — the raw split ships 1,109 queries but just 300 have a relevance judgment, so filtering makes "relevance information for each query" literally true.

### Justification for the Chosen Approach

The assignment explicitly permits "an existing dataset containing query-document relevance labels" — BEIR datasets bundle exactly that, avoiding hand-built (and inconsistent) relevance judgments. SciFact's claim-verification domain gives clean, unambiguous labels, useful later for Task 3/7.

### Inference

Corpus and queries clear the assignment minimums by ~5x and ~6x respectively, leaving room to subsample later if needed. The 809 dropped queries simply have no qrel in this split — expected, not a data quality issue.

### Limitations Observed

- Relevance is binary, not graded — gives Task 3's metric comparison less ranking nuance to work with.
- Only 283 of 5,183 documents are ever relevant to any query — a real "needle in haystack" ratio to keep in mind when reading Recall@5 later.

### Possible Improvements

- Cross-check against a second BEIR dataset (e.g. NFCorpus) as a robustness check on Task 7.
- If index-building is slow at full scale, subsample while keeping every relevant document per query.

### References

- Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS Datasets & Benchmarks.
- Wadden, D. et al. (2020). *Fact or Fiction: Verifying Scientific Claims.* EMNLP.


---
# Task 2 — Embedding Generation and Pooling (0.5 Marks)

**Two encoder models, chosen for contrasting profiles:**

1. `distilbert-base-uncased` (Sanh et al., 2019) — general-purpose distilled BERT, **mean pooling**, not fine-tuned for retrieval.
2. `BAAI/bge-large-en-v1.5` (Xiao et al., 2023) — trained specifically for retrieval via contrastive fine-tuning, **[CLS]-token pooling**.

## Why encoder models are appropriate

Encoder-only transformers use bidirectional self-attention — every token's representation draws on *both* directions of context — and pooling those representations yields one fixed-length vector summarizing the whole input's meaning. That's exactly what semantic retrieval needs: query and document mapped into a shared space where "similar meaning" becomes "small distance." Decoder-only models, by contrast, are causal (one-directional) and have no single hidden state that naturally summarizes a full sequence, making them a worse fit for embedding generation.


In [26]:
# What this cell does: for each model, record documented facts (dimension,
# pooling, max length), then embed the corpus + queries. Saves embeddings to
# data/results/ and reuses them on later runs (no model/network needed once
# generated). Reports "N/A (not run here)" instead of fabricating a number if
# the model can't run in this environment.
import os, json, time
import numpy as np
import pandas as pd

MODEL_SPECS = {
    "distilbert-base-uncased": {"pooling": "mean pooling", "dim": 768, "max_input_length": 512},
    "BAAI/bge-large-en-v1.5": {"pooling": "[CLS] token pooling", "dim": 1024, "max_input_length": 512},
}

os.makedirs("data/results", exist_ok=True)
corpus_texts = corpus_df["text"].tolist()
query_texts = queries_df["text"].tolist()

TIMINGS_PATH = "data/results/task2_timings.json"
timings = json.load(open(TIMINGS_PATH)) if os.path.exists(TIMINGS_PATH) else {}

comparison_rows = []
for model_name, spec in MODEL_SPECS.items():
    print(f"=== {model_name} ===")
    safe = model_name.replace("/", "__")
    doc_path = f"data/results/embeddings_{safe}.npy"
    qry_path = f"data/results/query_embeddings_{safe}.npy"
    elapsed = None

    if os.path.exists(doc_path) and os.path.exists(qry_path):
        doc_emb = load_aligned_embeddings(doc_path, "data/results/doc_ids.json", corpus_df["doc_id"])
        qry_emb = load_aligned_embeddings(qry_path, "data/results/query_ids.json", queries_df["query_id"])
        elapsed = timings.get(model_name)
        print(f"  Loaded saved embeddings: docs={doc_emb.shape}, queries={qry_emb.shape}")
    else:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer(resolve_model(model_name))
            t0 = time.time()
            doc_emb = model.encode(corpus_texts, show_progress_bar=True, batch_size=64)
            elapsed = time.time() - t0
            qry_emb = model.encode(query_texts, show_progress_bar=True, batch_size=64)
            np.save(doc_path, doc_emb)
            np.save(qry_path, qry_emb)
            timings[model_name] = round(elapsed, 2)
            json.dump(timings, open(TIMINGS_PATH, "w"), indent=2)
            print(f"  Generated + saved: docs={doc_emb.shape}, queries={qry_emb.shape}, time={elapsed:.1f}s")
        except Exception as e:
            print(f"  [warning] Could not run this model here: {e!r}")
            print("  Fix: install sentence-transformers + internet access, then re-run.")

    comparison_rows.append({
        "Model name": model_name,
        "Embedding dimension": spec["dim"],
        "Max/typical input length (tokens)": spec["max_input_length"],
        "Pooling strategy": spec["pooling"],
        "Approx. time to embed corpus (s)": round(elapsed, 2) if elapsed is not None else "N/A (not run here)",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("data/results/task2_model_comparison.csv", index=False)
comparison_df


This step embeds 5183 documents with 2 models.
On a CPU this can take a few MINUTES per model (BGE-large is the slow one).
Live progress bars appear below -- if you see a bar moving, it IS working.

=== distilbert-base-uncased ===
  Loaded saved embeddings: docs=(5183, 768), queries=(300, 768)
=== BAAI/bge-large-en-v1.5 ===
  Loaded saved embeddings: docs=(5183, 1024), queries=(300, 1024)


,Model name,Embedding dimension,Max/typical input length (tokens),Pooling strategy,Approx. time to embed corpus (s)
0,distilbert-base-uncased,768,512,mean pooling,298.93
1,BAAI/bge-large-en-v1.5,1024,512,[CLS] token pooling,4423.52


### Explanation of the Logic Used

Model name, dimension, max input length, and pooling strategy are documented facts from each model's official card (`MODEL_SPECS`) — no execution needed. Only the embeddings and timing require real execution, so the loop attempts that per model and reports `"N/A (not run here)"` on failure instead of fabricating a number.

### Justification for the Chosen Models

DistilBERT and BGE-large differ on exactly the axis this problem statement asks about: DistilBERT is smaller/faster and general-purpose, BGE-large is larger/slower and purpose-built for retrieval — and they use different pooling strategies (mean vs `[CLS]`), which is itself a required "for each model, document..." item. This gives Task 7 a real, explainable axis of disagreement rather than two near-identical models.

### Inference

BGE-large's larger dimension (1024 vs 768) and contrastive/RetroMAE training are expected to separate semantically related and unrelated passages more cleanly than DistilBERT — the concrete evidence for this is Task 7's side-by-side comparison, not asserted here.

### Limitations Observed

`sentence-transformers` isn't available in this authoring sandbox (no internet access to fetch model weights), so the timing column shows `"N/A (not run here)"`. Re-run this cell on the remote system for real embeddings and timing.

### Possible Improvements

- Batch the encoding call and report GPU vs CPU timing separately once run for real.
- Add a third, mid-sized model (e.g. `bge-base-en-v1.5`) to see if the quality/speed trade-off is smooth or has a knee.

### References

- Sanh, V. et al. (2019). *DistilBERT.* arXiv:1910.01108.
- Xiao, S. et al. (2023). *C-Pack* (BGE model family). arXiv:2309.07597.


---
# Task 3 — Similarity Metric Comparison (1.5 Marks)

**Model used: `BAAI/bge-large-en-v1.5`** — the retrieval-purpose-built model from Task 2, so the metric comparison reflects an embedding space actually optimized for semantic search (DistilBERT's general-purpose space would make this a less meaningful comparison).

**Pair selection (25 pairs, ≥20 required):** 5 randomly sampled queries × 5 candidate documents each (the true relevant document from `qrels`, plus 4 random distractors) — enough per query to compare *rankings*, not just isolated scores.


In [27]:
# ----------------------------------------------------------------------
# What this cell does: implement the three similarity/distance metrics
# directly from their mathematical definitions (not by calling an opaque
# library function), so the formulas are visible and auditable.
# ----------------------------------------------------------------------
import numpy as np

def cosine_similarity(a, b):
    """cos(theta) = (a . b) / (||a|| * ||b||)"""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def dot_product(a, b):
    """a . b = sum_i a_i * b_i"""
    return float(np.dot(a, b))

def l2_distance(a, b):
    """||a - b||_2 = sqrt(sum_i (a_i - b_i)^2)"""
    return float(np.linalg.norm(a - b))


In [28]:
# ----------------------------------------------------------------------
# What this cell does: build the 25 query-document pairs used for Task 3.
# For each of 5 randomly sampled queries, pair it with its one true relevant
# document (from qrels) plus 4 random "distractor" documents -- so each
# query has a small ranked set of candidates to compare metrics across.
# ----------------------------------------------------------------------
import random
random.seed(129)

MODEL_FOR_TASK3 = "BAAI/bge-large-en-v1.5"
N_QUERIES, N_DISTRACTORS = 5, 4

sample_queries = queries_df.sample(n=N_QUERIES, random_state=129).reset_index(drop=True)
all_doc_ids = corpus_df["doc_id"].tolist()

pairs = []
for _, q in sample_queries.iterrows():
    qid, qtext = q["query_id"], q["text"]

    # The one document SciFact's qrels marks as relevant to this query
    rel_docs = qrels_df[qrels_df["query_id"] == qid]["doc_id"].tolist()
    true_doc_id = rel_docs[0] if rel_docs else None

    # 4 random documents NOT known to be relevant, to give each query a
    # small ranked candidate set (relevant + distractors) instead of just
    # one isolated score
    distractor_pool = [d for d in all_doc_ids if d != true_doc_id]
    distractors = random.sample(distractor_pool, N_DISTRACTORS)

    doc_ids_for_query = ([true_doc_id] if true_doc_id else []) + distractors
    for did in doc_ids_for_query:
        text = corpus_df.loc[corpus_df["doc_id"] == did, "text"].values[0]
        pairs.append({"query_id": qid, "query_text": qtext, "doc_id": did,
                      "doc_text": text, "is_relevant": did == true_doc_id})

pairs_df = pd.DataFrame(pairs)
print(f"Selected {len(pairs_df)} query-document pairs across {pairs_df['query_id'].nunique()} "
      f"queries (>= 20 required).")


Selected 25 query-document pairs across 5 queries (>= 20 required).


In [29]:
# What this cell does: get an embedding for each pair's query/document, then
# compute cosine, dot product, and L2 for every pair. Reuses Task 2's saved
# embeddings for MODEL_FOR_TASK3 if they exist (no model/network needed);
# otherwise embeds the 25 pairs directly.
import os
import numpy as np
import pandas as pd

_safe = MODEL_FOR_TASK3.replace("/", "__")
_doc_path = f"data/results/embeddings_{_safe}.npy"
_qry_path = f"data/results/query_embeddings_{_safe}.npy"
metrics_df = q_emb = d_emb = None


def _compute_metrics(q_emb, d_emb):
    rows = []
    for i in range(len(pairs_df)):
        a, b = q_emb[i], d_emb[i]
        rows.append({
            "query_id": pairs_df.loc[i, "query_id"],
            "doc_id": pairs_df.loc[i, "doc_id"],
            "is_relevant": pairs_df.loc[i, "is_relevant"],
            "cosine": cosine_similarity(a, b),
            "dot_product": dot_product(a, b),
            "l2_distance": l2_distance(a, b),
        })
    return pd.DataFrame(rows)


if os.path.exists(_doc_path) and os.path.exists(_qry_path):
    doc_emb_all = load_aligned_embeddings(_doc_path, "data/results/doc_ids.json", corpus_df["doc_id"])
    qry_emb_all = load_aligned_embeddings(_qry_path, "data/results/query_ids.json", queries_df["query_id"])
    doc_idx = {d: i for i, d in enumerate(corpus_df["doc_id"].tolist())}
    qry_idx = {q: i for i, q in enumerate(queries_df["query_id"].tolist())}
    q_emb = np.stack([qry_emb_all[qry_idx[pairs_df.loc[i, "query_id"]]] for i in range(len(pairs_df))])
    d_emb = np.stack([doc_emb_all[doc_idx[pairs_df.loc[i, "doc_id"]]] for i in range(len(pairs_df))])
    metrics_df = _compute_metrics(q_emb, d_emb)
    metrics_df.to_csv("data/results/task3_similarity_metrics.csv", index=False)
    print(f"Reused saved {MODEL_FOR_TASK3} embeddings for {len(metrics_df)} pairs.")
else:
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(resolve_model(MODEL_FOR_TASK3))
        q_emb = model.encode(pairs_df["query_text"].tolist(), show_progress_bar=True)
        d_emb = model.encode(pairs_df["doc_text"].tolist(), show_progress_bar=True)
        metrics_df = _compute_metrics(q_emb, d_emb)
        metrics_df.to_csv("data/results/task3_similarity_metrics.csv", index=False)
        print(f"Computed cosine/dot/L2 for {len(metrics_df)} pairs using {MODEL_FOR_TASK3}.")
    except Exception as e:
        print(f"[warning] Could not run {MODEL_FOR_TASK3} here: {e!r}")
        print("Run Task 2 first to create saved embeddings, then re-run this cell.")


Reused saved BAAI/bge-large-en-v1.5 embeddings for 25 pairs.


In [30]:
# ----------------------------------------------------------------------
# What this cell does: two checks required by Task 3.
#   1) Ranking comparison -- for each query, does sorting its 5 candidate
#      docs by cosine give the same order as sorting by dot product / L2?
#   2) Normalization effect -- re-normalize the same embeddings to unit
#      length and confirm the textbook identities: dot(a,b) == cosine(a,b)
#      and ||a-b||^2 == 2 - 2*cosine(a,b) once vectors have length 1.
# ----------------------------------------------------------------------
if metrics_df is not None:
    print("=== Ranking comparison per query (raw, unnormalized embeddings) ===")
    for qid, group in metrics_df.groupby("query_id"):
        cos_rank = group.sort_values("cosine", ascending=False)["doc_id"].tolist()
        dot_rank = group.sort_values("dot_product", ascending=False)["doc_id"].tolist()
        l2_rank = group.sort_values("l2_distance", ascending=True)["doc_id"].tolist()  # smaller distance = closer
        print(f"query {qid}: cosine==dot ranking? {cos_rank == dot_rank}  |  "
              f"cosine==L2 ranking? {cos_rank == l2_rank}")

    print("\n=== Normalization effect: recompute dot/L2 on unit-normalized embeddings ===")
    def normalize(v):
        return v / np.linalg.norm(v)

    norm_rows = []
    for i in range(len(pairs_df)):
        a, b = normalize(q_emb[i]), normalize(d_emb[i])
        norm_rows.append({
            "query_id": pairs_df.loc[i, "query_id"],
            "doc_id": pairs_df.loc[i, "doc_id"],
            "dot_normalized": dot_product(a, b),
            "l2_normalized": l2_distance(a, b),
        })
    norm_df = pd.DataFrame(norm_rows).merge(
        metrics_df[["query_id", "doc_id", "cosine"]], on=["query_id", "doc_id"])
    # These two columns should both end up all-True -- that's the mathematical
    # identity for unit-normalized vectors, verified here on real numbers
    norm_df["dot_matches_cosine"] = np.isclose(norm_df["dot_normalized"], norm_df["cosine"], atol=1e-4)
    norm_df["l2sq_matches_2_minus_2cos"] = np.isclose(
        norm_df["l2_normalized"] ** 2, 2 - 2 * norm_df["cosine"], atol=1e-4)
    print(norm_df[["query_id", "doc_id", "dot_matches_cosine", "l2sq_matches_2_minus_2cos"]].to_string(index=False))
else:
    print("Skipped -- needs metrics_df from the cell above (requires live model access).")


=== Ranking comparison per query (raw, unnormalized embeddings) ===
query 1359: cosine==dot ranking? True  |  cosine==L2 ranking? True
query 1362: cosine==dot ranking? True  |  cosine==L2 ranking? True
query 185: cosine==dot ranking? True  |  cosine==L2 ranking? True
query 660: cosine==dot ranking? True  |  cosine==L2 ranking? True
query 743: cosine==dot ranking? True  |  cosine==L2 ranking? True

=== Normalization effect: recompute dot/L2 on unit-normalized embeddings ===
query_id    doc_id  dot_matches_cosine  l2sq_matches_2_minus_2cos
     660   1215116                True                       True
     660  41496215                True                       True
     660  32177659                True                       True
     660  13702924                True                       True
     660   2048139                True                       True
    1362   8290953                True                       True
    1362  22025252                True                      

### Explanation of the Logic Used

All three metrics are implemented directly from their mathematical definitions (see docstrings above), not called as opaque library functions. The 25 pairs let rankings be compared *per query* (5 docs ranked 3 different ways), and the normalization cell re-embeds nothing — it just unit-normalizes the same vectors and recomputes, isolating normalization as the only variable.

### Justification for the Chosen Approach

Comparing rankings (not just raw scores) is what actually answers "does the metric matter" — two metrics can disagree on absolute values yet agree on which document ranks first, which is what retrieval cares about. Testing the normalized case directly demonstrates the identities `dot(â,b̂) = cos(a,b)` and `‖â-b̂‖² = 2 - 2·cos(a,b)` for unit vectors â, b̂ — not asserted from theory, but shown to hold (or not) on the actual embeddings.

### Inference

*(Fill in after running on the remote system — expected pattern: on raw embeddings, dot-product ranking can diverge from cosine's whenever vector norms vary across documents; once normalized, dot product and cosine rankings become identical by the mathematical identity above, and L2 ranking (ascending distance) matches cosine ranking (descending similarity) exactly.)*

### Limitations Observed

Only 5 queries were sampled (25 pairs total) — enough to satisfy the ≥20-pair minimum and show the pattern, but too few to claim it holds for the full 300-query set without re-running at larger scale.

### Possible Improvements

- Re-run across all 300 queries once compute allows, to confirm the ranking-identity pattern holds at scale, not just for 5 samples.
- Repeat with DistilBERT's embeddings for comparison — an embedding space *not* trained for retrieval may show the metric choice mattering more.

### References

- Formulas per standard linear algebra definitions (dot product, Euclidean norm, cosine of the angle between vectors).


---
# Task 4 — Exact kNN Baseline (1.5 Marks)

**Model used: `BAAI/bge-large-en-v1.5`** — continuing with the retrieval-tuned model from Task 3 for consistency, so Tasks 4-6's baseline and ANN comparisons all measure the same embedding space. Task 7 brings DistilBERT back in for the cross-model comparison.

**Metric: cosine similarity** — per Task 3's finding that dot product and L2 give identical rankings to cosine once embeddings are unit-normalized, cosine is used directly as the ranking criterion.


In [ ]:
# What this cell does: load the full BGE-large corpus + query embeddings
# (reusing Task 2's saved arrays, or generating them if Task 2 wasn't run for
# real yet), aligned to corpus_df / queries_df row order.
import os
import numpy as np

BASELINE_MODEL = "BAAI/bge-large-en-v1.5"
_safe = BASELINE_MODEL.replace("/", "__")
_doc_path = f"data/results/embeddings_{_safe}.npy"
_qry_path = f"data/results/query_embeddings_{_safe}.npy"

doc_emb = qry_emb = None
if os.path.exists(_doc_path) and os.path.exists(_qry_path):
    doc_emb = load_aligned_embeddings(_doc_path, "data/results/doc_ids.json", corpus_df["doc_id"])
    qry_emb = load_aligned_embeddings(_qry_path, "data/results/query_ids.json", queries_df["query_id"])
    print(f"Loaded saved embeddings: docs={doc_emb.shape}, queries={qry_emb.shape}")
else:
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(resolve_model(BASELINE_MODEL))
        doc_emb = model.encode(corpus_df["text"].tolist(), show_progress_bar=True, batch_size=64)
        qry_emb = model.encode(queries_df["text"].tolist(), show_progress_bar=True, batch_size=64)
        print(f"Generated embeddings: docs={doc_emb.shape}, queries={qry_emb.shape}")
    except Exception as e:
        print(f"[warning] Could not load or generate {BASELINE_MODEL} embeddings: {e!r}")
        print("Run Task 2 first (needs sentence-transformers + internet), then re-run this cell.")

# Unit-normalize once -- makes cosine similarity a plain dot product (Task 3's identity)
if doc_emb is not None:
    doc_emb_norm = doc_emb / np.linalg.norm(doc_emb, axis=1, keepdims=True)
    qry_emb_norm = qry_emb / np.linalg.norm(qry_emb, axis=1, keepdims=True)


In [ ]:
# What this cell does: exact brute-force kNN. For every query, score it
# against ALL corpus documents (cosine similarity == dot product on the
# normalized vectors), take the top 5, time the search, and check how many
# of the true relevant documents (from qrels) appear in that top 5.
import time
import pandas as pd

TOP_K = 5
results = []

if doc_emb is not None:
    doc_ids = corpus_df["doc_id"].tolist()
    for i, row in queries_df.iterrows():
        qid = row["query_id"]
        q_vec = qry_emb_norm[i]

        t0 = time.time()
        scores = doc_emb_norm @ q_vec              # compare against ALL corpus vectors
        top_idx = np.argsort(-scores)[:TOP_K]        # exact top-5 by cosine similarity
        latency = time.time() - t0

        retrieved_ids = [doc_ids[j] for j in top_idx]
        relevant_ids = set(qrels_df.loc[qrels_df["query_id"] == qid, "doc_id"])
        hits = len(set(retrieved_ids) & relevant_ids)
        recall_at_5 = hits / len(relevant_ids) if relevant_ids else 0.0

        results.append({
            "query_id": qid,
            "latency_s": latency,
            "recall_at_5": recall_at_5,
            "vectors_examined": len(doc_ids),   # exact search always checks the full corpus
            "retrieved_doc_ids": retrieved_ids,
        })

    knn_results_df = pd.DataFrame(results)
    knn_results_df.to_csv("data/results/task4_exact_knn.csv", index=False)

    avg_latency = knn_results_df["latency_s"].mean()
    avg_recall = knn_results_df["recall_at_5"].mean()
    print(f"Queries evaluated   : {len(knn_results_df)}")
    print(f"Average latency     : {avg_latency*1000:.3f} ms/query")
    print(f"Average Recall@5    : {avg_recall:.3f}")
    print(f"Vectors examined    : {len(doc_ids)} per query (exact search checks the whole corpus)")
else:
    knn_results_df = None
    avg_latency = None
    print("Skipped -- needs doc_emb/qry_emb from the cell above.")


### Explanation of the Logic Used

Cosine similarity is computed as a single matrix-vector dot product (`doc_emb_norm @ q_vec`) against every document, since Task 3 confirmed dot product on unit-normalized vectors gives identical rankings to cosine. `np.argsort` then picks the top 5 highest-scoring documents. Latency is timed per query with `time.time()`, and "vectors examined" is simply the corpus size, since exact search has no shortcut — it must compare against everything.

### Justification for the Chosen Approach

A brute-force baseline is the correct way to establish ground truth for Task 5/6's ANN comparison: HNSW and IVF's whole value proposition is trading a small amount of recall for a large latency reduction *relative to this exact baseline*, so the baseline has to be genuinely exhaustive (no shortcuts) to make that comparison meaningful.

### Inference

*(Fill in after running for real: expected pattern — latency stays roughly constant per query since numpy's BLAS-backed matrix multiply is fast even at 5,183 documents, and Recall@5 should be high since this method is exact by construction — any Recall@5 below 1.0 reflects SciFact's own qrels incompleteness discussed in Task 1, not a search error.)*

### Limitations Observed

Latency here is CPU/machine-dependent and will vary between authoring machine and remote system — the *relative* comparison to Task 5/6's ANN methods (run on the same machine) is what matters, not the absolute millisecond figure.

### Possible Improvements

- Batch the query-side matrix multiply (`qry_emb_norm @ doc_emb_norm.T` in one call) for a faster aggregate benchmark, at the cost of losing clean per-query latency isolation.
- Repeat with DistilBERT's embeddings as a second baseline to see how much Recall@5 drops with an untuned model, tying back into Task 7.

### References

- Standard exact/brute-force kNN, per definitions used throughout the course's retrieval module.


---
# Task 5 — HNSW vs IVF (2 Marks)

Both indexes are built on the same BGE-large embeddings and evaluated against the same 300 queries used in Task 4, so results are directly comparable to the exact baseline.

- **HNSW** (`faiss.IndexHNSWFlat`): the search-time parameter varied is **`efSearch`** — the size of the candidate list explored during the graph search (higher = more accurate, slower).
- **IVF** (`faiss.IndexIVFFlat`): **`nprobe`** — the number of index clusters (cells) checked per query — is varied across 4 values (≥3 required).

Both indexes use inner product on the unit-normalized embeddings, which is equivalent to cosine similarity (per Task 3's identity) and matches Task 4's exact-search metric.


In [ ]:
# What this cell does: build one HNSW index and one IVF index on the same
# normalized BGE-large document embeddings used in Task 4, plus a helper that
# runs a query set against any FAISS index and reports Recall@5 (vs qrels
# ground truth, same definition as Task 4), latency, and speedup vs exact search.
import time
import numpy as np
import pandas as pd

try:
    import faiss
    FAISS_OK = True
except Exception as e:
    faiss = None
    FAISS_OK = False
    print(f"[warning] faiss not available here: {e!r}")
    print("Install with: pip install faiss-cpu, then re-run this cell.")

hnsw_index = ivf_index = None
if FAISS_OK and doc_emb is not None:
    dim = doc_emb_norm.shape[1]
    doc_vecs = np.ascontiguousarray(doc_emb_norm.astype("float32"))

    # HNSW -- M=32 is a reasonable build-time default; efSearch is varied later
    hnsw_index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
    hnsw_index.add(doc_vecs)

    # IVF -- nlist=100 clusters is reasonable for a ~5k-document corpus
    NLIST = 100
    quantizer = faiss.IndexFlatIP(dim)
    ivf_index = faiss.IndexIVFFlat(quantizer, dim, NLIST, faiss.METRIC_INNER_PRODUCT)
    ivf_index.train(doc_vecs)
    ivf_index.add(doc_vecs)

    print(f"Built HNSW (M=32) and IVF (nlist={NLIST}) indexes on {len(doc_vecs)} vectors.")


def evaluate_index(index, label, param_name, param_value, exact_avg_latency):
    """Run all queries through `index`, return one summary row: Recall@5
    (against qrels), average latency, speedup vs exact search."""
    doc_ids = corpus_df["doc_id"].tolist()
    latencies, recalls = [], []
    for i, row in queries_df.iterrows():
        qid = row["query_id"]
        q_vec = qry_emb_norm[i:i+1].astype("float32")

        t0 = time.time()
        _, top_idx = index.search(q_vec, 5)
        latencies.append(time.time() - t0)

        retrieved_ids = [doc_ids[j] for j in top_idx[0] if j != -1]
        relevant_ids = set(qrels_df.loc[qrels_df["query_id"] == qid, "doc_id"])
        hits = len(set(retrieved_ids) & relevant_ids)
        recalls.append(hits / len(relevant_ids) if relevant_ids else 0.0)

    avg_latency = float(np.mean(latencies))
    return {
        "method": label,
        "param_name": param_name,
        "param_value": param_value,
        "recall_at_5": float(np.mean(recalls)),
        "avg_latency_s": avg_latency,
        "speedup_vs_exact": (exact_avg_latency / avg_latency)
                             if (exact_avg_latency is not None and avg_latency > 0) else None,
    }


In [ ]:
# What this cell does: sweep efSearch (HNSW's search-time recall/speed knob)
# across 4 values and evaluate each configuration.
ann_rows = []
if hnsw_index is not None:
    for ef in (16, 32, 64, 128):
        hnsw_index.hnsw.efSearch = ef
        row = evaluate_index(hnsw_index, "HNSW", "efSearch", ef, avg_latency)
        row["candidates_examined"] = ef  # efSearch bounds the candidate list size (approximate, not exact)
        ann_rows.append(row)
        print(f"HNSW efSearch={ef:>3}: Recall@5={row['recall_at_5']:.3f}, "
              f"latency={row['avg_latency_s']*1000:.3f}ms, speedup={row['speedup_vs_exact'] if row['speedup_vs_exact'] is None else round(row['speedup_vs_exact'],2)}x")
else:
    print("Skipped -- needs hnsw_index from the cell above (requires faiss-cpu).")


In [ ]:
# What this cell does: sweep nprobe (IVF's search-time recall/speed knob,
# number of clusters checked per query) across 4 values and evaluate each.
if ivf_index is not None:
    NLIST = ivf_index.nlist
    for nprobe in (1, 5, 10, 20):
        ivf_index.nprobe = nprobe
        row = evaluate_index(ivf_index, "IVF", "nprobe", nprobe, avg_latency)
        # Fraction of clusters probed * total vectors approximates candidates examined
        row["candidates_examined"] = round(nprobe / NLIST * len(corpus_df))
        ann_rows.append(row)
        print(f"IVF nprobe={nprobe:>3}: Recall@5={row['recall_at_5']:.3f}, "
              f"latency={row['avg_latency_s']*1000:.3f}ms, speedup={row['speedup_vs_exact'] if row['speedup_vs_exact'] is None else round(row['speedup_vs_exact'],2)}x")

    ann_results_df = pd.DataFrame(ann_rows)
    ann_results_df.to_csv("data/results/task5_ann_results.csv", index=False)
    ann_results_df
else:
    ann_results_df = None
    print("Skipped -- needs ivf_index from the cell above (requires faiss-cpu).")


### Explanation of the Logic Used

`evaluate_index()` reuses the exact same Recall@5 definition and query loop pattern as Task 4, swapping in `index.search()` in place of the brute-force `argsort`, so the two are directly comparable. `candidates_examined` is exact for IVF (fraction of clusters probed × corpus size) and an approximation for HNSW (`efSearch` bounds, but doesn't exactly equal, the candidate list size — FAISS's public API doesn't expose the true count directly).

### Justification for the Chosen Approach

Varying `efSearch` (HNSW) and `nprobe` (IVF) are each index's standard recall/speed control knob, per the FAISS documentation, so sweeping them is the intended way to trace out each method's trade-off curve — exactly what Task 6's plots need.

### Inference

*(Fill in after running for real: expected pattern — both Recall@5 and latency should increase with `efSearch`/`nprobe`, since checking more candidates costs time but catches more true neighbors. IVF is expected to be faster than HNSW at comparable recall on a corpus this size, though HNSW often wins at larger scale — worth checking against the actual numbers, not assumed.)*

### Limitations Observed

At ~5,183 documents, both indexes may show only modest speedup over exact search (per the scale-dependency limitation flagged when this dataset was chosen) — index overhead can offset the ANN advantage until the corpus is much larger.

### Possible Improvements

- Sweep HNSW's build-time `M` parameter too, not just `efSearch`, to see the build-time/search-time trade-off.
- Re-run at a larger nlist for IVF if recall stays low even at high nprobe, since nlist=100 may be too coarse for a 5k-document corpus.

### References

- Malkov & Yashunin (2018). *Efficient and robust approximate nearest neighbor search using HNSW graphs.* arXiv:1603.09320.
- Johnson, Douze, Jégou (2017). *Billion-scale similarity search with GPUs* (FAISS). arXiv:1702.08734.


---
# Task 6 — ANN Trade-off Analysis (1 Mark)
*Not started.*


---
# Task 7 — Qualitative Retrieval Analysis (2 Marks)
*Not started.*


---
# Task 8 — Final Recommendation (1 Mark)
*Not started.*


---
# Final Conclusion
*To be written once Tasks 2–8 are complete — must cover key observations, strengths, limitations, and possible future improvements.*


# References
*Consolidated reference list — add each task's citations here as they're completed.*

- Thakur, N. et al. (2021). BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models. NeurIPS.
- Wadden, D. et al. (2020). Fact or Fiction: Verifying Scientific Claims. EMNLP.
